# **1. Разведочный анализ данных (Exploratory Data Analysis): исследование структуры и поведения набора данных Metro Interstate Traffic Volume**

* __Цель разведочного анализа данных:__ исследовать структуру, качество и временные закономерности подготовленного набора данных.
* __Задачи разведочного анализа данных:__ 
  - оценить размерность, схему и временной диапазон данных;
  - диагностировать пропуски, дубликаты и разрывы временной оси;
  - исследовать распределение целевой переменной `traffic_volume`;
  - проанализировать суточные, недельные и месячные профили нагрузки;
  - оценить автокорреляционную структуру для обоснования последующего выбора лагов.
* __Алгоритм выполнения:__
  1. __Загрузка данных:__ использование модуля `load_raw_data()` из слоя Data Ingestion для получения подготовленного почасового набора данных.
  2. __Диагностика качества:__ оценка пропущенных значений, дубликатов, регулярности временной сетки и состава признаков.
  3. __Анализ целевой переменной:__ расчет описательных статистик, построение распределения и визуализация временной динамики `traffic_volume`.
  4. __Анализ временных закономерностей:__ расчет средних профилей транспортной нагрузки по часам, дням недели, месяцам и типу дня.
  5. __Автокорреляционный анализ:__ расчет ACF/PACF и значений автокорреляции для настроенных лагов `LAG_HOURS`.
  6. __Интерпретация результатов:__ обобщение выявленных закономерностей и формулирование выводов для последующего этапа feature engineering.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from traffic_forecasting.config import (
    DATETIME_COLUMN,
    LAG_HOURS,
    REPORTS_DIR,
    TARGET_COLUMN,
    TIME_FREQUENCY,
)
from traffic_forecasting.data_loader import load_raw_data

FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)

## **1.1. Загрузка подготовленных данных (Load Prepared Raw Data)**

In [ ]:
# Use the project ingestion layer to keep EDA consistent with the training pipeline.
data = load_raw_data()

## **1.2. Обзор структуры набора данных (Dataset Overview)**

In [ ]:
hourly_grid_rows = int(data[TARGET_COLUMN].isna().sum())
dataset_summary = pd.DataFrame(
    {
        "row_count": [len(data)],
        "column_count": [data.shape[1]],
        "date_min": [data[DATETIME_COLUMN].min()],
        "date_max": [data[DATETIME_COLUMN].max()],
        "full_duplicate_rows": [int(data.duplicated().sum())],
        "duplicate_timestamps": [int(data.duplicated(DATETIME_COLUMN).sum())],
        "hourly_grid_rows_without_target": [hourly_grid_rows],
    }
)
dataset_summary.to_csv(TABLES_DIR / "dataset_summary.csv", index=False)

display(Markdown("### **Сводная характеристика набора данных (Dataset Summary)**"))
display(dataset_summary)
display(Markdown("### **Типы колонок (Column Types)**"))
display(pd.DataFrame({"column": data.columns, "dtype": data.dtypes.astype(str)}))
display(Markdown("### **Первые строки набора данных (First Rows)**"))
display(data.head())
display(Markdown("### **Последние строки набора данных (Last Rows)**"))
display(data.tail())

## **1.3. Диагностика качества данных (Data Quality Diagnostics)**

In [ ]:
missing_values_summary = pd.DataFrame(
    {
        "missing_count": data.isna().sum(),
        "missing_percentage": data.isna().mean().mul(100),
    }
).sort_values("missing_count", ascending=False)
missing_values_summary.to_csv(TABLES_DIR / "missing_values_summary.csv")

display(Markdown("### **Сводка пропущенных значений (Missing Values Summary)**"))
display(missing_values_summary)
print(f"Full duplicate rows: {data.duplicated().sum():,}")
print(f"Duplicate timestamps: {data.duplicated(DATETIME_COLUMN).sum():,}")
print(f"Hourly grid rows without target: {hourly_grid_rows:,}")

## **1.4. Описательная статистика (Descriptive Statistics)**

In [ ]:
numeric_summary = data.describe(include="number").T
categorical_columns = data.select_dtypes(exclude="number").columns.difference([DATETIME_COLUMN])
categorical_summary = data[categorical_columns].describe().T
traffic_volume_summary = data[TARGET_COLUMN].describe().to_frame("traffic_volume")

display(Markdown("### **Описательная статистика числовых колонок (Numeric Statistics)**"))
display(numeric_summary)
display(Markdown("### **Описательная статистика категориальных колонок (Categorical Statistics)**"))
display(categorical_summary)
display(Markdown("### **Статистика целевой переменной (Target Statistics)**"))
display(traffic_volume_summary)

## **1.5. Анализ целевой переменной (Target Variable Analysis)**

In [ ]:
# Drop missing targets only for descriptive plots, without modifying the dataset.
target_values = data[TARGET_COLUMN].dropna()

display(
    Markdown("### **Распределение и boxplot целевой переменной (Target Distribution and Boxplot)**")
)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(target_values, bins=50, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Traffic Volume Distribution")
axes[0].set_xlabel("Traffic volume (vehicles/hour)")
axes[0].set_ylabel("Observation count")

sns.boxplot(x=target_values, ax=axes[1], color="lightsteelblue")
axes[1].set_title("Traffic Volume Boxplot")
axes[1].set_xlabel("Traffic volume (vehicles/hour)")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "traffic_volume_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
display(Markdown("### **Временной ряд транспортной нагрузки (Traffic Volume Time Series)**"))
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(data[DATETIME_COLUMN], data[TARGET_COLUMN], linewidth=0.5, color="steelblue")
ax.set_title("Hourly Traffic Volume Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Traffic volume (vehicles/hour)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "traffic_volume_time_series.png", dpi=150, bbox_inches="tight")
plt.show()

## **1.6. Анализ временных профилей (Temporal Profile Analysis)**

In [ ]:
# Calendar fields are temporary EDA labels and are not saved as model features.
profile_data = data.loc[data[TARGET_COLUMN].notna(), [DATETIME_COLUMN, TARGET_COLUMN]].copy()
profile_data["hour"] = profile_data[DATETIME_COLUMN].dt.hour
profile_data["day_of_week"] = profile_data[DATETIME_COLUMN].dt.day_name()
profile_data["month"] = profile_data[DATETIME_COLUMN].dt.month
profile_data["is_weekend"] = profile_data[DATETIME_COLUMN].dt.dayofweek >= 5

hourly_profile = profile_data.groupby("hour", as_index=False)[TARGET_COLUMN].mean()
monthly_profile = profile_data.groupby("month", as_index=False)[TARGET_COLUMN].mean()
weekend_profile = profile_data.groupby("is_weekend", as_index=False)[TARGET_COLUMN].mean()

WEEKDAY_ORDER = (
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
)
daily_profile = (
    profile_data.groupby("day_of_week", as_index=False)[TARGET_COLUMN]
    .mean()
    .set_index("day_of_week")
    .reindex(WEEKDAY_ORDER)
    .reset_index()
)

display(Markdown("### **Таблица среднего почасового профиля (Hourly Profile Table)**"))
display(hourly_profile)

display(Markdown("### **Таблица профиля по дням недели (Day-of-Week Profile Table)**"))
display(daily_profile)

display(Markdown("### **Таблица месячного профиля (Monthly Profile Table)**"))
display(monthly_profile)

display(Markdown("### **Сравнение будних и выходных дней (Weekday vs Weekend Profile)**"))
display(weekend_profile)

In [ ]:
display(Markdown("### **Средний почасовой профиль (Hourly Traffic Profile)**"))
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=hourly_profile, x="hour", y=TARGET_COLUMN, marker="o", ax=ax)
ax.set_title("Average Traffic Volume by Hour")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Average traffic volume (vehicles/hour)")
ax.set_xticks(range(24))
fig.tight_layout()
fig.savefig(FIGURES_DIR / "traffic_volume_by_hour.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
display(Markdown("### **Профиль по дням недели (Day-of-Week Profile)**"))
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(
    data=daily_profile,
    x="day_of_week",
    y=TARGET_COLUMN,
    order=WEEKDAY_ORDER,
    ax=ax,
)
ax.set_title("Average Traffic Volume by Day of Week")
ax.set_xlabel("Day of week")
ax.set_ylabel("Average traffic volume (vehicles/hour)")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "traffic_volume_by_day_of_week.png", dpi=150, bbox_inches="tight")
plt.show()

display(Markdown("### **Месячный профиль (Monthly Profile)**"))
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=monthly_profile, x="month", y=TARGET_COLUMN, marker="o", ax=ax)
ax.set_title("Average Traffic Volume by Month")
ax.set_xlabel("Month")
ax.set_ylabel("Average traffic volume (vehicles/hour)")
ax.set_xticks(range(1, 13))
fig.tight_layout()
fig.savefig(FIGURES_DIR / "traffic_volume_by_month.png", dpi=150, bbox_inches="tight")
plt.show()

## **1.7. Диагностика автокорреляции временного ряда (Time-Series Autocorrelation Diagnostics)**

In [ ]:
# Preserve hourly positions so configured lags correspond to real time offsets.
target_series = data.set_index(DATETIME_COLUMN)[TARGET_COLUMN].asfreq(TIME_FREQUENCY)
selected_autocorrelation = pd.DataFrame(
    {
        "lag_hours": LAG_HOURS,
        "autocorrelation": [target_series.autocorr(lag=lag) for lag in LAG_HOURS],
    }
)
selected_autocorrelation.to_csv(
    TABLES_DIR / "autocorrelation_selected_lags.csv",
    index=False,
)
display(Markdown("### **Автокорреляция выбранных лагов (Selected Lag Autocorrelation)**"))
display(selected_autocorrelation)

In [ ]:
segment_ids = target_series.notna().ne(target_series.notna().shift()).cumsum()
continuous_segments = [
    segment.dropna() for _, segment in target_series.groupby(segment_ids) if segment.notna().all()
]
# Use the longest continuous segment to avoid interpolating the target variable.
pacf_series = max(continuous_segments, key=len)

display(Markdown("### **Графики ACF и PACF (ACF/PACF Plots)**"))
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
plot_acf(
    target_series,
    lags=max(LAG_HOURS),
    zero=False,
    missing="conservative",
    ax=axes[0],
)
axes[0].set_title("Traffic Volume Autocorrelation (up to 168 Hours)")
axes[0].set_xlabel("Lag (hours)")
axes[0].set_ylabel("Autocorrelation")

plot_pacf(pacf_series, lags=max(LAG_HOURS), zero=False, method="ywm", ax=axes[1])
axes[1].set_title("Traffic Volume Partial Autocorrelation")
axes[1].set_xlabel("Lag (hours)")
axes[1].set_ylabel("Partial autocorrelation")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "traffic_volume_acf_pacf.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"PACF continuous segment length: {len(pacf_series):,} hours")

## **1.8. Анализ и интерпретация результатов разведочного анализа данных (Analysis and Interpretation of EDA Results)**

На этапе разведочного анализа был исследован подготовленный набор данных Metro Interstate Traffic Volume, используемый для решения задачи прогнозирования почасовой транспортной нагрузки.

**Ключевые результаты:**
1. __Подготовленный набор данных имеет корректную временную структуру.__  
   После выполнения слоя Data Ingestion данные представлены в виде регулярного почасового временного ряда. Полностью дублирующиеся строки и дубликаты временных меток отсутствуют.
2. __Пропущенные значения имеют различную природу и не должны обрабатываться одинаково.__  
   Пропуски в поле `holiday` интерпретируются как обычные непраздничные часы, а не как потеря данных. В то же время пропуски в `traffic_volume`, погодных и категориальных признаках соответствуют строкам, добавленным при восстановлении регулярной почасовой сетки.
3. __Целевая переменная `traffic_volume` обладает выраженной неоднородностью распределения.__  
   Распределение транспортной нагрузки не является равномерным: присутствуют периоды низкой, средней и высокой интенсивности движения. Это указывает на нелинейный характер зависимости целевой переменной от временных, календарных, погодных и исторических факторов.
4. __Во временной динамике транспортной нагрузки наблюдаются устойчивые суточные закономерности.__  
   Средняя транспортная нагрузка минимальна в ночные часы и резко возрастает утром. Наибольшие значения наблюдаются в дневные и вечерние часы, что соответствует типичному поведению городской и пригородной транспортной системы.
5. __Недельный профиль показывает различие между рабочими и выходными днями.__  
   Средняя транспортная нагрузка в будние дни выше, чем в выходные. Это подтверждает влияние календарного фактора на интенсивность движения и обосновывает использование признаков `day_of_week` и `is_weekend` на этапе feature engineering.
6. __Месячные профили указывают на наличие сезонной компоненты.__  
   Средняя нагрузка изменяется по месяцам, что может отражать сезонные особенности транспортного спроса, погодных условий и календарной активности.
7. __Автокорреляционный анализ подтверждает значимость исторических значений транспортной нагрузки.__  
   Высокие значения автокорреляции на краткосрочных, суточных и недельных лагах подтверждают наличие зависимости текущей нагрузки от предыдущих наблюдений и обосновывают использование набора лагов `LAG_HOURS = (1, 2, 3, 24, 168)` на этапе формирования признакового пространства.

__Итоговое методологическое резюме:__ разведочный анализ подтвердил пригодность подготовленного набора данных для решения задачи регрессионного прогнозирования `traffic_volume` и показал наличие выраженных временных, календарных и автокорреляционных закономерностей, которые должны быть учтены при построении моделей машинного обучения.